# 겹치는 선분의 길이

https://school.programmers.co.kr/learn/courses/30/lessons/120876

**문제 설명**  
선분 3개가 주어질 때, 두 개 이상의 선분이 겹치는 부분의 길이를 반환  

**제한사항**  
- lines의 원소는 [a, b] 형태이며 a < b  
- -100 ≤ a, b ≤ 100  
- 선분이 점으로만 겹치는 경우는 길이에 포함하지 않음

---
## 내 답안

In [ ]:
def solution(lines):

    end_list = [set(), set(), set()]
    for i, line in enumerate(lines):
        for x in range(line[0]+1, line[1]+1):
            end_list[i].add(x)

    duplicated = (end_list[0] & end_list[1]) | (end_list[1] & end_list[2]) | (end_list[0] & end_list[2])
    answer = len(duplicated)
    return answer

print(solution([[0, 1], [2, 5], [3, 9]]))   # 2
print(solution([[-1, 1], [1, 3], [3, 9]]))  # 0
print(solution([[0, 5], [3, 9], [1, 10]]))  # 8

---
## 내 답안 리뷰

**핵심 아이디어 — 단위 구간을 오른쪽 끝점 정수로 표현**

`range(a+1, b+1)` 은 정수 집합 `{a+1, a+2, ..., b}` 를 만든다.  
정수 x는 단위 구간 `[x-1, x]` 을 대표한다고 해석하면,  
선분 `[a, b]` 가 커버하는 모든 단위 구간이 정확히 이 집합에 담긴다.

```
선분 [2, 5]  →  {3, 4, 5}  →  구간 [2,3], [3,4], [4,5] 표현
선분 [3, 9]  →  {4, 5, 6, 7, 8, 9}  →  구간 [3,4], [4,5], ..., [8,9] 표현
교집합 → {4, 5}  →  겹치는 구간 [3,4], [4,5]  →  길이 2 ✓
```

집합 연산 `&` (교집합) 으로 겹치는 구간을 구하고, `|` (합집합) 으로 세 쌍의 겹침을 모두 모은다. **정확하고 영리한 접근.**

**개선할 점**

| 항목 | 현재 | 개선 |
|---|---|---|
| 주석 처리된 디버그 코드 | `# print(...)` 가 여러 줄 남아 있음 | 제출 전 삭제 |
| 하드코딩된 리스트 크기 | `[set(), set(), set()]` — 항상 3개 | `[set() for _ in lines]` 로 유연하게 |
| 집합 생성 방식 | `for x in ...: .add(x)` | `set(range(...))` 으로 한 줄 단축 가능 |

**점으로만 겹치는 케이스가 자동으로 걸러지는 이유**  
`range(a+1, b+1)` 은 단위 구간(길이 1)의 오른쪽 끝만 저장하므로,  
예를 들어 `[−1,1]` 과 `[1,3]` 이 점 1에서 만나더라도,  
두 집합의 교집합에는 점 1을 오른쪽 끝으로 하는 구간 `[0,1]` 과 — 잠깐, 실제로 확인해 보면:
- `[-1, 1]` → `range(0, 2)` = {0, 1}
- `[1, 3]`  → `range(2, 4)` = {2, 3}
- 교집합 = {} → 길이 0 ✓

선분이 점 1에서만 만나는데 교집합이 비어 있는 이유:  
정수 1은 `[-1,1]` 에서 `[0,1]` 구간을, `[1,3]` 에서 `[1,2]` 구간을 나타내어 서로 다른 구간이기 때문이다.

---
## 다른 사람 풀이

### 풀이 1: 오프셋 배열로 커버리지 카운팅 (가장 많이 선택된 방식)

**아이디어**  
좌표 범위 −100 ~ 100을 크기 200짜리 배열로 표현한다 (`인덱스 = 좌표 + 100`).  
각 선분이 커버하는 단위 구간에 +1씩 누적하고, 2 이상인 칸의 수를 센다.

```
선분 [2, 5] → range(2+100, 5+100) → 인덱스 102~104 에 +1
선분 [3, 9] → range(3+100, 9+100) → 인덱스 103~108 에 +1
cnt[103], cnt[104] = 2  →  겹치는 구간 2개  →  길이 2 ✓
```

**내 풀이와 비교**  
개념은 동일하지만, 세 쌍의 집합 연산 대신 배열 합산 하나로 해결한다.  
선분 개수가 늘어도 코드 변경 없이 그대로 동작하는 것이 장점.

In [ ]:
# 풀이 1: 오프셋 배열
def solution(lines):
    cnt = [0] * 200  # 인덱스 i → 실제 좌표 i - 100
    for a, b in lines:
        for i in range(a + 100, b + 100):  # 단위 구간 [a,a+1],...,[b-1,b] 에 +1
            cnt[i] += 1
    return sum(1 for c in cnt if c >= 2)

print(solution([[0, 1], [2, 5], [3, 9]]))   # 2
print(solution([[-1, 1], [1, 3], [3, 9]]))  # 0
print(solution([[0, 5], [3, 9], [1, 10]]))  # 8

### 풀이 2: 포함-배제 원리 (Inclusion-Exclusion)

**아이디어**  
두 선분 A, B의 겹치는 길이 = `max(0, min(b1,b2) − max(a1,a2))`  
이를 3쌍 모두에 적용하되, 3개 모두 겹치는 구간은 두 번 빼야 한다.

```
정확히 2개 겹치는 점  →  쌍 교집합 합산에서 1번 카운트  ✓
3개 모두 겹치는 점    →  쌍 교집합 합산에서 3번 카운트  →  −2×|A∩B∩C| 로 보정

결과 = |A∩B| + |A∩C| + |B∩C| − 2×|A∩B∩C|
```

**장점**: 루프 없이 O(1), 수학적으로 가장 명확  
**단점**: 선분 개수가 정확히 3개일 때만 공식 적용 가능

In [ ]:
# 풀이 2: 포함-배제 원리
def solution(lines):
    def overlap(l1, l2):
        return max(0, min(l1[1], l2[1]) - max(l1[0], l2[0]))

    a, b, c = lines

    # 세 쌍의 겹치는 길이 합산
    pair_total = overlap(a, b) + overlap(a, c) + overlap(b, c)

    # 세 선분 모두 겹치는 구간: A∩B 를 먼저 구하고 C 와 교집합
    ab = [max(a[0], b[0]), min(a[1], b[1])]
    triple = overlap(ab, c) if ab[0] < ab[1] else 0

    return pair_total - 2 * triple

print(solution([[0, 1], [2, 5], [3, 9]]))   # 2
print(solution([[-1, 1], [1, 3], [3, 9]]))  # 0
print(solution([[0, 5], [3, 9], [1, 10]]))  # 8

---
## 풀이 비교

| 풀이 | 시간복잡도 | 특징 |
|---|---|---|
| 내 답안 (집합 교집합) | O(200 × N) | 집합 연산 활용, 핵심 아이디어 동일 |
| 풀이 1 (오프셋 배열) | O(200 × N) | 가장 보편적인 스윕 방식, N개 선분에도 확장 가능 |
| 풀이 2 (포함-배제) | O(1) | 수학 공식 기반, N=3 고정일 때만 사용 가능 |